In [1]:
import pandas as pd

In [2]:
df_tweets = pd.read_csv('all_musk_posts.csv')
print(df_tweets.shape)
print(df_tweets.columns.tolist())
print(df_tweets.head())
print(df_tweets['createdAt'].dtype)

(55099, 24)
['id', 'url', 'twitterUrl', 'fullText', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'createdAt', 'bookmarkCount', 'isReply', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'inReplyToUsername', 'isPinned', 'isRetweet', 'isQuote', 'isConversationControlled', 'possiblySensitive', 'quoteId', 'quote', 'retweet']
                    id                                                url  \
0  1655159652990976000  https://x.com/elonmusk/status/1655159652990976000   
1  1657261624867299339  https://x.com/elonmusk/status/1657261624867299339   
2  1623774484795920384  https://x.com/elonmusk/status/1623774484795920384   
3  1656900119202254854  https://x.com/elonmusk/status/1656900119202254854   
4  1616531874763116544  https://x.com/elonmusk/status/1616531874763116544   

                                          twitterUrl  \
0  https://twitter.com/elonmusk/status/1655159652...   
1  https://twitter.com/elonmusk/status/1657261624...   
2  https://twitte

/var/folders/p1/dpyr3mh148b4f4w13br8mfph0000gn/T/ipykernel_28098/3562563520.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tweets = pd.read_csv('all_musk_posts.csv')


In [3]:
print(df_tweets.tail ())

                        id                                                url  \
55094  1863748322424402395  https://x.com/elonmusk/status/1863748322424402395   
55095  1863740336331170304  https://x.com/elonmusk/status/1863740336331170304   
55096  1863740028406337896  https://x.com/elonmusk/status/1863740028406337896   
55097  1863736644773220593  https://x.com/elonmusk/status/1863736644773220593   
55098  1863736436945555943  https://x.com/elonmusk/status/1863736436945555943   

                                              twitterUrl  \
55094  https://twitter.com/elonmusk/status/1863748322...   
55095  https://twitter.com/elonmusk/status/1863740336...   
55096  https://twitter.com/elonmusk/status/1863740028...   
55097  https://twitter.com/elonmusk/status/1863736644...   
55098  https://twitter.com/elonmusk/status/1863736436...   

                                        fullText  retweetCount  replyCount  \
55094                                  @alx Cool          89.0       150.0

In [4]:
#Cleaning the data

#Reloading with low_memory=False
df_tweets = pd.read_csv('all_musk_posts.csv', low_memory=False)

#Extracting only the columns we need
df_tweets = df_tweets[['fullText', 'createdAt', 'isRetweet', 'isReply', 'likeCount', 'retweetCount']]

#converting createdAt attribute from object to datetime
df_tweets['createdAt'] = pd.to_datetime(df_tweets['createdAt'], utc=True)

#Extracting just the date and removing time to make it align with TSLA daily price data
df_tweets['date'] = df_tweets['createdAt'].dt.date

# Filtering out retweets so as to only keep Elon's original thoughts
df_tweets = df_tweets[df_tweets['isRetweet'] != True]

#Sort chronologically
df_tweets = df_tweets.sort_values('createdAt').reset_index(drop=True)

print(df_tweets.shape)
print(df_tweets.head())

(54007, 7)
                                            fullText  \
0  Please ignore prior tweets, as that was someon...   
1  Went to Iceland on Sat to ride bumper cars on ...   
2  I made the volume on the Model S http://t.co/w...   
3  Great Voltaire quote, arguably better than Twa...   
4                  That was a total non sequitur btw   

                  createdAt isRetweet isReply  likeCount  retweetCount  \
0 2010-06-04 18:31:57+00:00     False   False     6392.0         697.0   
1 2011-12-01 09:55:11+00:00     False   False      197.0          27.0   
2 2011-12-01 10:29:04+00:00     False   False      124.0          19.0   
3 2011-12-03 08:20:28+00:00     False   False       87.0          36.0   
4 2011-12-03 08:22:07+00:00     False   False      133.0          12.0   

         date  
0  2010-06-04  
1  2011-12-01  
2  2011-12-01  
3  2011-12-03  
4  2011-12-03  


In [5]:
#Installing FinBERT for sentiment analysis
!pip install transformers torch

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Loading FinBERT
tokenizer = BertTokenizer.from_pretrained('ProsusAI/finbert')
model_finbert = BertForSequenceClassification.from_pretrained('ProsusAI/finbert')
model_finbert.eval()

In [ ]:
def get_finbert_sentiment(text):
    try:
        inputs = tokenizer(
            str(text), 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding=True
        )
        with torch.no_grad():
            outputs = model_finbert(**inputs)
        
        scores = torch.softmax(outputs.logits, dim=1).numpy()[0]
        # FinBERT returns [positive, negative, neutral]
        compound = scores[0] - scores[1]  # positive - negative
        label = ['positive', 'negative', 'neutral'][np.argmax(scores)]
        return compound, label
    except:
        return 0.0, 'neutral'

In [ ]:
# Applying to TSLA tweets
print("Running FinBERT sentiment analysis...")
df_tsla_tweets[['sentiment_score', 'sentiment_label']] = df_tsla_tweets['fullText'].apply(
    lambda x: pd.Series(get_finbert_sentiment(x))
)
print("Done!")
print(df_tsla_tweets[['fullText', 'sentiment_score', 'sentiment_label']].head(10))
print("\nSentiment Distribution:")
print(df_tsla_tweets['sentiment_label'].value_counts())

In [ ]:
#Extracting TSLA relevant tweets
tsla_keywords = [
    'tesla', 'tsla', 'model s', 'model 3', 'model x', 'model y',
    'cybertruck', 'electric', 'ev', 'autopilot', 'fsd', 'giga',
    'production', 'delivery', 'stock', 'short', 'shareholders',
    'battery', 'supercharger', 'roadster', 'semi', 'powerwall'
]

pattern = '|'.join(tsla_keywords)
df_tsla_tweets = df_tweets[
        df_tweets['fullText'].str.lower().str.contains(pattern, na=False)
].copy()

print(f"Total tweets: {len(df_tweets)}")
print(f"TSLA-relevant tweets: {len(df_tsla_tweets)}")
print(f"Percentage: {len(df_tsla_tweets)/len(df_tweets)*100:,.1f}%")

In [ ]:
#Filling NaN engagement values with 1 so as to treat them normally when assigning weight
df_tsla_tweets['likeCount'] = df_tsla_tweets['likeCount'].fillna(1)
df_tsla_tweets['retweetCount'] = df_tsla_tweets['retweetCount'].fillna(1)

#Retweets weighted 2x more since they spread further
df_tsla_tweets['engagement'] = df_tsla_tweets['likeCount'] + df_tsla_tweets['retweetCount']*2

#weighted sentiment = sentiment x engagement
df_tsla_tweets['weighted_sentiment'] = df_tsla_tweets['sentiment_score']*df_tsla_tweets['engagement']

print(df_tsla_tweets[['fullText', 'sentiment_score', 'engagement', 'weighted_sentiment']].head(10))

In [ ]:
#Aggregating by date
daily_sentiment_tsla = df_tsla_tweets.groupby('date').agg(
    avg_sentiment = ('sentiment_score', 'mean'),
    weighted_sentiment = ('weighted_sentiment', 'sum'),
    total_engagement = ('engagement', 'sum'),
    tweet_count = ('sentiment_score', 'count'),
    positive_count = ('sentiment_label', lambda x: (x == 'positive').sum()),
    negative_count = ('sentiment_label', lambda x: (x == 'negative').sum())
).reset_index()

print(daily_sentiment_tsla.shape)
print(daily_sentiment_tsla.head(10))

In [ ]:
import numpy as np

df_tsla = pd.read_csv('df_tsla.csv', index_col='Date', parse_dates=True)
print(df_tsla.shape)
print(df_tsla.columns.tolist())
print(df_tsla.head())

In [ ]:
#Merging with TSLA price data
daily_sentiment_tsla['date'] = pd.to_datetime(daily_sentiment_tsla['date'])
df_tsla.index = pd.to_datetime(df_tsla.index)

In [ ]:
#Merge
df_combined = df_tsla.merge(daily_sentiment_tsla, left_index=True, right_on='date', how='left')
df_combined = df_combined.set_index('date')

In [ ]:
#Check missing sentiment days
print("Shape: ", df_combined.shape)
print("\nMissing sentiment values: ", df_combined['avg_sentiment'].isna().sum())
print("\nSamples:")
df_combined.head(10)

In [ ]:
# Forward fill — carry last known sentiment forward
# Logic: market still "remembers" the last tweet's impact
df_combined['avg_sentiment'] = df_combined['avg_sentiment'].fillna(method='ffill')
df_combined['weighted_sentiment'] = df_combined['weighted_sentiment'].fillna(method='ffill')
df_combined['total_engagement'] = df_combined['total_engagement'].fillna(method='ffill')
df_combined['tweet_count'] = df_combined['tweet_count'].fillna(0)  # 0 tweets that day is accurate
df_combined['positive_count'] = df_combined['positive_count'].fillna(0)
df_combined['negative_count'] = df_combined['negative_count'].fillna(0)

# Check remaining NaNs
print("Missing values after fill:")
print(df_combined.isnull().sum())
print("\nShape:", df_combined.shape)

In [ ]:
# Backfill the remaining NaNs at the start
df_combined['avg_sentiment'] = df_combined['avg_sentiment'].fillna(method='bfill')
df_combined['weighted_sentiment'] = df_combined['weighted_sentiment'].fillna(method='bfill')
df_combined['total_engagement'] = df_combined['total_engagement'].fillna(method='bfill')

print("Missing values after backfill:")
print(df_combined[['avg_sentiment', 'weighted_sentiment', 'total_engagement']].isnull().sum())

In [ ]:
# Checking what years have the most TSLA-relevant tweets
df_tsla_tweets['year'] = pd.to_datetime(df_tsla_tweets['date']).dt.year
print(df_tsla_tweets.groupby('year')['fullText'].count())

## Rebuilding LSTM Model

In [ ]:
features_with_sentiment = [
    'Close', 'MA_10', 'MA_20', 'MA_50', 'Daily_Return',
    'Volatility', 'RSI', 'MACD', 'MACD_Signal',
    'BB_Upper', 'BB_Lower',
    'avg_sentiment', 'weighted_sentiment', 'total_engagement'
]

data_sentiment = df_combined[features_with_sentiment].values
print("Data shape: ", data_sentiment.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_sentiment = MinMaxScaler()
scaled_sentiment = scaler_sentiment.fit_transform(data_sentiment)
print("Scaled shape: ", scaled_sentiment.shape)

In [ ]:
#Creating sequences
def create_sequences(data, sequence_length=60):
    X, y = [],[]
    for i in range(sequence_length, len(data)):
        X.append(data[i-sequence_length: i])
        y.append(data[i, 0])
    return np.array(X), np.array(y)
X_sent, y_sent = create_sequences(scaled_sentiment)
print("X shape: ", X_sent.shape)
print("y shape: ", y_sent.shape)

In [ ]:
#Train/Test split
split = int(0.8*len(X_sent))
X_train_s, X_test_s = X_sent[:split], X_sent[split:]
y_train_s, y_test_s = y_sent[:split], y_sent[split:]

print("X_train shape: ", X_train_s.shape)
print("X_test shape: ", X_test_s.shape)

In [ ]:
#Building the LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model_sentiment = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train_s.shape[1], X_train_s.shape[2])),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_sentiment.compile(optimizer='adam', loss='mean_squared_error')
model_sentiment.summary()

In [ ]:
#Traning the model
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history_sentiment = model_sentiment.fit(
    X_train_s, y_train_s,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
#Predictions and metrics
predictions_s = model_sentiment.predict(X_test_s)

# Inverse transform
pred_full_s = np.zeros((len(predictions_s), scaled_sentiment.shape[1]))
pred_full_s[:, 0] = predictions_s[:, 0]
predicted_prices_s = scaler_sentiment.inverse_transform(pred_full_s)[:, 0]

actual_full_s = np.zeros((len(y_test_s), scaled_sentiment.shape[1]))
actual_full_s[:, 0] = y_test_s
actual_prices_s = scaler_sentiment.inverse_transform(actual_full_s)[:, 0]

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse_s = np.sqrt(mean_squared_error(actual_prices_s, predicted_prices_s))
mae_s = mean_absolute_error(actual_prices_s, predicted_prices_s)
print(f"RMSE: ${rmse_s:.2f}")
print(f"MAE:  ${mae_s:.2f}")

In [ ]:
#Directional Accuracy
actual_dir_s = np.diff(actual_prices_s) > 0
predicted_dir_s = np.diff(predicted_prices_s) > 0
dir_acc_s = np.mean(actual_dir_s == predicted_dir_s) * 100
print(f"Directional Accuracy: {dir_acc_s:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.style.use("fivethirtyeight")

plt.figure(figsize=(15, 6))
plt.plot(actual_prices_s, label='Actual Price', linewidth=0.8)
plt.plot(predicted_prices_s, label='Predicted Price (with Sentiment)', linewidth=0.8)
plt.title('TSLA - Predicted vs Actual (With Sentiment)')
plt.xlabel('Days')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Saveing FinBERT LSTM model
model_sentiment.save('tsla_finbert_model.keras')

# Saveing FinBERT scaler
import pickle
with open('scaler_sentiment.pkl', 'wb') as f:
    pickle.dump(scaler_sentiment, f)